# 5.2 — Perturbation recall: within-batch vs across-batch

Held-out cross-set Recall@k on the same paired views (C / CD / S / SD) as §5.1, plus the cp_measure baseline (CPM). Two retrieval regimes:

- **Within-batch**: query and gallery are both from the held-out batch. Self is excluded via similarity-mask. Tests retrieval *without* batch effects to overcome — the achievable ceiling.
- **Across-batch**: query is the held-out batch, gallery is every other batch. Tests cross-batch generalisation — the real-world headline metric.

Held-out batch per dataset matches the DataModule split (val_frac=0.1, seed=42): JUMP `source_4`, RxRx1 `HEPG2-07`, Rxrx3C `plate_4`.

**The within − across gap quantifies the batch-correction tax**: how much retrieval performance is lost when generalising across batches. A small gap means the embedding is batch-invariant; a large gap means most of the retrieval signal is batch-specific.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

TEXTWIDTH_PT = 397.48499  # \the\textwidth
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27
FULLWIDTH_IN = TEXTWIDTH_IN
HALFWIDTH_IN = TEXTWIDTH_IN / 2

def neurips_style():
    mpl.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'mathtext.fontset': 'dejavusans',
        'font.size': 7,
        'axes.titlesize': 7,
        'axes.labelsize': 7,
        'xtick.labelsize': 7,
        'ytick.labelsize': 7,
        'legend.fontsize': 7,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.linewidth': 0.6,
        'xtick.major.width': 0.6,
        'ytick.major.width': 0.6,
        'lines.linewidth': 1.0,
        'patch.linewidth': 0.6,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.01,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
    })

neurips_style()

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

DL_RECALL_DIR  = Path('../01_perturbation_recall')
CPM_RECALL_CSV = Path('../baselines/perturbation_recall_within_and_across_cp_measure.csv')
FIG_DIR = Path('.')
FIG_DIR.mkdir(exist_ok=True)

DATASETS = ['RxRx1', 'Rxrx3C', 'JUMP']
ENCODERS = ['SubCell', 'DINO', 'OpenPhenom']

ENCODER_DISPLAY = {
    'SubCell':    'SubCell',
    'DINO':       'DINO\nViT-S',
    'OpenPhenom': 'Open\nPhenom',
    'CPM':        'CPM',
}

DL_RECALL_CSV = {
    'RxRx1':  DL_RECALL_DIR / 'perturbation_recall_within_across_rxrx1.csv',
    'Rxrx3C': DL_RECALL_DIR / 'perturbation_recall_within_across_rxrx3c.csv',
    'JUMP':   DL_RECALL_DIR / 'perturbation_recall_within_across_jump.csv',
}

DATASET_LABEL = {
    'RxRx1':  'RxRx1\n(siRNA, HEPG2)',
    'Rxrx3C': 'RxRx3-core\n(CRISPR-KO, HUVEC)',
    'JUMP':   'JUMP-CP\n(compounds, U2OS)',
}

# 'Rxrx1' (CSV's group prefix) -> 'RxRx1' (canonical key)
_DATASET_NORM = {'Rxrx1': 'RxRx1', 'Rxrx3C': 'Rxrx3C', 'JUMP': 'JUMP'}

In [ ]:
VIEWS = ['C', 'CD', 'S', 'SD']
K_MAX = 10

frames = []
for ds, path in DL_RECALL_CSV.items():
    df = pd.read_csv(path)
    df['dataset'] = ds
    df['encoder'] = df['group'].str.split(' — ').str[1]
    frames.append(df)

# CPM: prefix-derive dataset, harmonise to RxRx1 key, encoder='CPM'
cpm = pd.read_csv(CPM_RECALL_CSV)
cpm['dataset'] = cpm['group'].str.split(' — ').str[0].map(_DATASET_NORM)
cpm['encoder'] = 'CPM'
frames.append(cpm)

raw = pd.concat(frames, ignore_index=True)
raw = raw[raw['embedding'].isin(['pre_harmony', 'post_harmony'])]
raw = raw[raw['recall_type'].isin(['within_batch', 'across_batch'])]
raw = raw[raw['view'].isin(VIEWS) | (raw['encoder'] == 'CPM')]
raw.head()

In [ ]:
# Headline metric is R@10. Pivot to (dataset, encoder, view, embedding, recall_type) -> R@10.
wide = raw.pivot_table(
    index=['dataset', 'encoder', 'view', 'embedding'],
    columns='recall_type',
    values='R@10',
).reset_index()
wide['delta_batch'] = wide['within_batch'] - wide['across_batch']  # batch-correction tax

dataset_order = pd.Categorical(wide['dataset'], categories=DATASETS, ordered=True)
wide = wide.assign(dataset=dataset_order)
wide = wide.sort_values(['embedding', 'dataset', 'encoder', 'view']).reset_index(drop=True)
wide

In [ ]:
# Per-stage summary: dataset-mean R@10 within and across batch (DL encoders only)
for stage in ['post_harmony', 'pre_harmony']:
    df_stage = wide[(wide['embedding'] == stage) & (wide['encoder'].isin(ENCODERS))]
    means = df_stage.groupby('dataset', observed=True)[['within_batch', 'across_batch', 'delta_batch']].mean().round(3)
    print(f'-- {stage} (DL encoder mean R@10) --')
    print(means)
    print()

In [ ]:
VIEW_COLOR = {
    'C':  '#2271b2',
    'CD': '#7fceaf',
    'S':  '#e84b4b',
    'SD': '#ff913a',
}
VIEWS_ORDERED = ['C', 'CD', 'S', 'SD']
CPM_COLOR = '#888888'

bar_w = 1.0
group_inner_w = len(VIEWS_ORDERED) * bar_w
group_gap = 1.5
group_step = group_inner_w + group_gap

encoder_centers = np.array([ei * group_step + (group_inner_w - bar_w) / 2 for ei in range(len(ENCODERS))])
cpm_x = len(ENCODERS) * group_step
group_centers_all = np.concatenate([encoder_centers, [cpm_x]])
group_labels_all = list(ENCODERS) + ['CPM']

bar_x = {
    (enc, view): ei * group_step + vi * bar_w
    for ei, enc in enumerate(ENCODERS)
    for vi, view in enumerate(VIEWS_ORDERED)
}


def label_panel(ax, label, x=-0.15, y=1.2):
    ax.text(x, y, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='top', ha='left')


def render_recall_figure(stage: str, out_stem: str, ymax: float | None = None):
    """2 rows (within / across) x 3 cols (datasets) of paired-view R@10 bars + CPM."""
    df_stage = wide[wide['embedding'] == stage]
    if ymax is None:
        ymax = float(df_stage[['within_batch', 'across_batch']].values.max()) * 1.05

    fig, axes = plt.subplots(2, 3, figsize=(FULLWIDTH_IN, 4), sharey=True)
    recall_rows = [('within_batch', 'Within-batch R@10'),
                   ('across_batch', 'Across-batch R@10')]

    for row_i, (recall_type, ylabel) in enumerate(recall_rows):
        for col_i, ds in enumerate(DATASETS):
            ax = axes[row_i, col_i]
            sub = df_stage[df_stage['dataset'] == ds].set_index(['encoder', 'view'])

            for enc in ENCODERS:
                for view in VIEWS_ORDERED:
                    val = sub.loc[(enc, view), recall_type] if (enc, view) in sub.index else np.nan
                    ax.bar(
                        bar_x[(enc, view)], val,
                        width=bar_w, color=VIEW_COLOR[view],
                        edgecolor='black', linewidth=0.4,
                    )

            cpm_val = float(sub.loc[('CPM', 'S'), recall_type]) if ('CPM', 'S') in sub.index else np.nan
            ax.bar(cpm_x, cpm_val, width=bar_w, color=CPM_COLOR, edgecolor='black', linewidth=0.4)
            ax.axhline(y=cpm_val, color='gray', linewidth=0.5, linestyle='--', zorder=0)

            all_x = [bar_x[(enc, v)] for enc in ENCODERS for v in VIEWS_ORDERED] + [cpm_x]
            ax.set_xticks(all_x)
            ax.set_xticklabels([''] * len(all_x))

            # Encoder labels only on the bottom row
            if row_i == 1:
                for centre, enc in zip(group_centers_all, group_labels_all):
                    ax.text(
                        centre, -0.10, ENCODER_DISPLAY[enc],
                        transform=ax.get_xaxis_transform(),
                        ha='center', va='top',
                    )

            # Dataset titles only on the top row
            if row_i == 0:
                ax.set_title(DATASET_LABEL[ds])

            ax.axhline(0, color='gray', linewidth=0.5)
            ax.set_xlim(-bar_w, cpm_x + bar_w)
            ax.set_ylim(0, ymax)

        axes[row_i, 0].set_ylabel(ylabel)

    # Panel labels A..F
    for ax, lbl in zip(axes.flat, 'ABCDEF'):
        label_panel(ax, lbl)

    # Legend in JUMP within-batch panel (top-right)
    legend_handles = [Patch(color=VIEW_COLOR[v], label=v) for v in VIEWS_ORDERED]
    legend_handles.append(Patch(color=CPM_COLOR, label='CPM'))
    axes[0, 2].legend(
        handles=legend_handles, loc='upper right', frameon=False,
        fontsize=7, handlelength=1.0, handleheight=1.0,
        borderpad=0.1, labelspacing=0.1,
    )

    stage_label = 'Post-Harmony' if stage == 'post_harmony' else 'Pre-Harmony'
    fig.suptitle(
        f'Held-out perturbation Recall@10 by view, encoder, and dataset — {stage_label}',
        fontsize=8, y=1.02,
    )
    fig.tight_layout()

    out_pdf = FIG_DIR / f'{out_stem}.pdf'
    fig.savefig(out_pdf)
    fig.savefig(out_pdf.with_suffix('.png'), dpi=300)
    plt.show()
    print(f'Saved: {out_pdf}')


render_recall_figure('post_harmony', 'fig_recall_post_harmony')
render_recall_figure('pre_harmony',  'fig_recall_pre_harmony')

In [ ]:
# Markdown tables: R@10 per (dataset, encoder, view, recall_type) for both stages.
for stage in ['post_harmony', 'pre_harmony']:
    label = 'Post-Harmony' if stage == 'post_harmony' else 'Pre-Harmony'
    df_stage = wide[wide['embedding'] == stage].set_index(['dataset', 'encoder', 'view'])

    lines = [f'## {label} — Recall@10', '']
    lines.append('| dataset | encoder | view | within | across | Δ_batch |')
    lines.append('|---|---|---|---|---|---|')
    for ds in DATASETS:
        for enc in ENCODERS:
            for view in VIEWS:
                if (ds, enc, view) not in df_stage.index:
                    continue
                row = df_stage.loc[(ds, enc, view)]
                lines.append(
                    f'| {ds} | {enc} | {view} | {row["within_batch"]:.3f} | {row["across_batch"]:.3f} | {row["delta_batch"]:+.3f} |'
                )
        if (ds, 'CPM', 'S') in df_stage.index:
            row = df_stage.loc[(ds, 'CPM', 'S')]
            lines.append(
                f'| {ds} | CPM | S | {row["within_batch"]:.3f} | {row["across_batch"]:.3f} | {row["delta_batch"]:+.3f} |'
            )
    print('\n'.join(lines))
    print()

## §5.2 prose draft (skeleton)

Placeholder. Populate after inspecting numbers.

Likely beats:
- Within-batch R@10 is uniformly higher than across-batch (batch-correction tax is universal). Quantify the mean gap per dataset.
- Modality gradient: RxRx1 has highest within and across; Rxrx3C / JUMP much lower. Note any swap in ordering.
- Δ_bg(R@10) = R@10(C) - R@10(S) on the across-batch axis: should mirror the §5.1 mAP-Δ_bg story.
- CPM under-performs every DL encoder on across-batch R@10, but on within-batch it gets closer (handcrafted features are competitive when batch effects don't need to be overcome).